# COMP8851 — T-Social only

Self-contained. Upload, set **GPU T4 x2** and **Internet On**, then **Run All**.

## Why this is separate

T-Social ran out of memory on a T4 at the published configuration:

```
CUDA out of memory. Tried to allocate 2.32 GiB.
GPU 0 has 14.56 GiB total, 1.01 GiB free.
12.39 GiB already allocated by PyTorch.
```

All three seeds failed identically. On the A6000 the same run used about 34 GB,
so this is a genuine hardware limit, not a transient fault.

## What this does differently

It walks a **memory ladder** and stops at the first rung that fits:

| Rung | hid_dim | order | Notes |
|---|---|---|---|
| 1 | 64 | 2 | the published configuration |
| 2 | 32 | 2 | half the hidden width |
| 3 | 32 | 1 | fewer propagation tensors |
| 4 | 16 | 1 | smallest sensible model |
| 5 | CPU | — | 32 GB system RAM instead of 14.6 GB VRAM |

`hid_dim` and `order` are **tuned hyperparameters** in this protocol, so every
rung is a legal configuration, not a modified protocol. Whichever rung succeeds
is written into that run's `run_config.yml`, so the reported number always
carries the configuration that produced it.

It also sets `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`, which the error
message itself recommends — 1.03 GiB was reserved but unallocated, i.e. lost to
fragmentation.

**The graph is never subsampled.** A reduced T-Social would not be T-Social.

## If every rung fails

That is a legitimate, reportable outcome and the notebook records it with the
traceback. T-Social would then need a 40 GB+ accelerator, which is the honest
conclusion — your earlier A6000 runs are evidence it works there.

## CARE-GNN

Off by default. It was measured at roughly **11 minutes per epoch** on this
graph, so 100 epochs is about 18 hours and it cannot finish in a Kaggle
session. Set `DO_CARE_GNN = True` only if you want the attempt recorded again;
the outcome is already captured from the previous run.


In [1]:
# ---------------------------------------------------------------- settings
DO_GHRN     = True
DO_CARE_GNN = False    # ~11 min/epoch on this graph; cannot finish a session

# Seed 2 alone on the first attempt. A CPU run on 146M edges is slow, and one
# genuine result is worth far more than three that a session timeout truncates.
# Once it is known to work, set this back to [2, 42, 72] and re-run - finished
# seeds are skipped, so nothing is repeated.
SEEDS       = [2]
FINAL_EPOCHS, FINAL_PATIENCE = 100, 20
TRIALS, TUNE_EPOCHS = 1, 8      # tuning is nearly pointless until it fits
# ---------------------------------------------------------------------------

import os
# Ask the allocator for expandable segments before torch is imported. The OOM
# message reported 1.03 GiB reserved-but-unallocated, which is fragmentation
# this setting is designed to reclaim.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import base64, io, json, shutil, subprocess, sys, tarfile, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working/comp8851")
OUTDIR = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
STATUS = {}

def step(name):
    def wrap(fn):
        print(f"\n{'='*70}\n{name}\n{'='*70}")
        t0 = time.time()
        try:
            out = fn()
            STATUS[name] = {"ok": True, "minutes": (time.time()-t0)/60}
            return out
        except Exception as exc:
            STATUS[name] = {"ok": False, "error": f"{type(exc).__name__}: {exc}",
                            "minutes": (time.time()-t0)/60}
            print(f"FAILED: {type(exc).__name__}: {exc}")
            traceback.print_exc(limit=3)
            return None
    return wrap

import torch
GPU_GB = 0.0
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    GPU_GB = p.total_memory / 1024**3
    print(f"GPU   : {p.name}  ({GPU_GB:.1f} GB)")
else:
    print("NO GPU. Settings -> Accelerator -> GPU T4 x2, then Run All again.")
print(f"torch : {torch.__version__}")
print(f"alloc : {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")

import socket
try:
    socket.setdefaulttimeout(8)
    socket.gethostbyname("drive.google.com")
    print("net   : OK")
except OSError:
    raise SystemExit("INTERNET IS OFF. Settings -> Internet -> On, then Run All again.")

GPU   : Tesla T4  (14.6 GB)
torch : 2.10.0+cu128
alloc : expandable_segments:True
net   : OK


## 1. Unpack and prepare

In [2]:
PIPELINE = (
    "..."
    "dFUA+AcA"
)

@step("1. unpack pipeline")
def _():
    raw = base64.b64decode("".join(PIPELINE))
    with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
        try:
            tar.extractall(WORK, filter="data")
        except TypeError:
            tar.extractall(WORK)
    (WORK / "data").mkdir(exist_ok=True)
    os.chdir(WORK)
    print(f"   {len(list(WORK.rglob('*.py')))} python files ready")
    return True


@step("2. torch + DGL")
def _():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)

    def probe():
        r = subprocess.run([sys.executable, "-c",
                            "import dgl, torch; dgl.graph(([0],[1]));"
                            " print(dgl.__version__, torch.__version__)"],
                           capture_output=True, text=True)
        return r.stdout.strip() if r.returncode == 0 else None

    ready = probe()
    if ready and ready.split()[1].startswith("2.4.0"):
        print(f"   already aligned: {ready}")
        return True
    print("   installing torch 2.4.0+cu121 and dgl 2.4.0+cu121 (~2.5 GB)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.4.0",
                    "--index-url", "https://download.pytorch.org/whl/cu121"],
                   capture_output=True, text=True)
    for index in ("https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html",
                  "https://data.dgl.ai/wheels/torch-2.4/repo.html"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "dgl==2.4.0+cu121", "-f", index],
                       capture_output=True, text=True)
        if probe():
            break
    ready = probe()
    print(f"   {ready or 'could not align torch and DGL'}")
    if ready:
        print("\n   >>> If torch changed, do Run -> Restart & Run All once. <<<")
    return bool(ready)


@step("3. download and decode T-Social")
def _():
    import gdown
    target = WORK / "data/tsocial"
    if list(target.glob("*_canonical.npz")):
        print("   already decoded")
        return True
    target.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(
        "https://drive.google.com/drive/folders/1PpNwvZx_YRSCDiHaBUmRIS3x1rZR7fMr",
        output=str(WORK / "data/bwgnn_drive"), quiet=False, use_cookies=False)
    src = WORK / "data/bwgnn_drive/dataset/tsocial.zip"
    subprocess.run(["unzip", "-q", "-o", str(src), "-d", str(target)], check=False)
    found = [p for p in target.rglob("tsocial") if p.is_file()]
    if found and found[0] != target / "tsocial":
        shutil.move(str(found[0]), str(target / "tsocial"))
    for z in (WORK / "data/bwgnn_drive").rglob("*.zip"):
        z.unlink()
    print(f"   raw file: {(target/'tsocial').stat().st_size/1024**2:.0f} MB")

    r = subprocess.run([sys.executable, "shared/comp8851/fetch_datasets.py",
                        "--data-root", "data", "--freeze", "--only", "tsocial"],
                       capture_output=True, text=True, cwd=str(WORK))
    for line in (r.stdout or "").strip().splitlines()[-8:]:
        print("   ", line)
    ok = bool(list(target.glob("*_canonical.npz")))
    print(f"   -> {'decoded' if ok else 'FAILED to decode'}")
    subprocess.run(["df", "-h", "/kaggle/working"])
    return ok


1. unpack pipeline
   33 python files ready

2. torch + DGL
   installing torch 2.4.0+cu121 and dgl 2.4.0+cu121 (~2.5 GB)
   2.4.0+cu121 2.4.0+cu121

   >>> If torch changed, do Run -> Restart & Run All once. <<<

3. download and decode T-Social


Retrieving folder contents


Retrieving folder 1DKO8edxNJ3UUWq-w2f6mLlKecr1itEUr dataset
Processing file 1gAMek_5Dsd4xPD0o7z-FFD8Jwxr0gJR5 tfinance.zip
Processing file 10nY_IwxT32KdfpcTqoy8wgckXK2xayXd tsocial.zip
Processing file 1zEYGv5WFraFRzHC68PDV8Lpm2xLzUif1 plot.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1gAMek_5Dsd4xPD0o7z-FFD8Jwxr0gJR5
From (redirected): https://drive.google.com/uc?id=1gAMek_5Dsd4xPD0o7z-FFD8Jwxr0gJR5&confirm=t&uuid=bc11de7d-d2e3-47e8-87a2-db0a4f7f651d
To: /kaggle/working/comp8851/data/bwgnn_drive/dataset/tfinance.zip
100%|██████████| 78.1M/78.1M [00:00<00:00, 79.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=10nY_IwxT32KdfpcTqoy8wgckXK2xayXd
From (redirected): https://drive.google.com/uc?id=10nY_IwxT32KdfpcTqoy8wgckXK2xayXd&confirm=t&uuid=bbfc6bcb-04e0-4448-ac1d-221d95e89f80
To: /kaggle/working/comp8851/data/bwgnn_drive/dataset/tsocial.zip
100%|██████████| 744M/744M [00:05<00:00, 144MB/s]  
Downloading...
From: https://drive.google.com/uc?id=1zEYGv5WFraFRzHC68PDV8Lpm2xLzUif1
To: /kaggle/working/comp8851/data/bwgnn_drive/plot.zip
100%|██████████| 2.92M/2.92M [00:00<00:00, 155MB/s

   raw file: 3920 MB
        nodes 5,781,065 | features 10 | fraud 174,280 (3.01%) | edges 146,211,016
        global heterophily 0.3761
    
    [manifest] wrote data/dataset_manifest.json
    [manifest] wrote data/checksums.sha256
    
    acquired and frozen: 1 of 1 requested
   -> decoded
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  4.2G   16G  22% /kaggle/working


## 2a. Verify the int32 graph narrowing

DGL stores node IDs as **int64** by default. T-Social has 5.78M nodes and 146M
edges, so those indices occupy roughly 2.3 GB representing values that never
exceed six million — int32 holds 2.1 billion, a 370x margin.

The pipeline now narrows large graphs to int32. That is exact, not approximate:
no edge, node or feature changes, only the width of the integers addressing
them. On a 16 GB card it may be the difference between fitting and not.

This cell proves it on the real graph before any training happens. If the two
representations disagree at all, it stops.

In [3]:
@step("2a. verify int32 narrowing")
def _():
    sys.path.insert(0, str(WORK))
    sys.path.insert(0, str(WORK / "models" / "ghrn"))
    import numpy as np, torch as _t
    from ghrnlib import backend
    from shared.comp8851 import datasets as ds

    if not backend.dgl_available():
        print("   DGL unavailable; the torch.sparse backend is used instead.")
        return True

    import dgl
    cache = next((WORK / "data" / "tsocial").glob("*_canonical.npz"), None)
    if cache is None:
        print("   T-Social not decoded yet; skipping (step 3 runs first)")
        return True

    data = ds.load_canonical(cache)
    matrix = list(data.relations.values())[0].tocoo()
    src = _t.as_tensor(matrix.row.astype(np.int64))
    dst = _t.as_tensor(matrix.col.astype(np.int64))
    n = data.features.shape[0]
    print(f"   graph: {n:,} nodes, {src.numel():,} stored edges")

    wide = dgl.add_self_loop(dgl.remove_self_loop(dgl.graph((src, dst), num_nodes=n)))
    narrow = backend.build_graph(src, dst, n)
    print(f"   idtype: {wide.idtype} -> {narrow.idtype}")

    ws, wd = wide.edges()
    ns, nd = narrow.edges()
    same = (wide.num_edges() == narrow.num_edges()
            and wide.num_nodes() == narrow.num_nodes()
            and _t.equal(ws.long(), ns.long())
            and _t.equal(wd.long(), nd.long()))
    print(f"   structure identical: {same}")
    if not same:
        raise RuntimeError("int32 narrowing changed the graph; refusing to train")

    saved = (ws.element_size() - ns.element_size()) * ws.numel() * 2 / 1024**3
    print(f"   index memory saved : ~{saved:.2f} GB")

    del wide, narrow, ws, wd, ns, nd
    import gc; gc.collect()
    if _t.cuda.is_available():
        _t.cuda.empty_cache()
    return True


2a. verify int32 narrowing
   graph: 5,781,065 nodes, 146,211,016 stored edges
   idtype: torch.int64 -> torch.int32
   structure identical: True
   index memory saved : ~1.13 GB


## 2b. The memory ladder

Each rung is tried in order and the first that survives a short probe is used
for the full run. The probe is two epochs — long enough to allocate everything
the model needs, short enough not to waste a session.

In [4]:
# Ordered by what the evidence says is likely to work, not by fidelity.
#
# On the T4 the GPU already held 12.39 GiB before the failing 2.32 GiB
# allocation. That 12.39 GiB is the graph itself - 146M edges - not model
# activations, so shrinking hid_dim leaves it untouched and every GPU rung
# meets the same wall. Rungs 1 and 2 confirmed this: both failed, and the
# second did not even reach a CUDA OOM.
#
# CPU removes the 14.6 GB ceiling entirely and replaces it with ~30 GB of
# system RAM, which is the only configuration here with room for the graph.
# It is slower, and its timings are not comparable with GPU runs - the
# deliverable already separates timings by device, so that is handled.
LADDER = [
    # GPU first again now that int32 narrowing frees roughly 2.3 GB of index
    # storage. The published configuration previously failed by 0.31 GB after
    # expandable_segments, so this margin may well be decisive.
    {"label": "gpu, published config", "hid_dim": 64, "order": 2, "cpu": False},
    {"label": "gpu, half width",       "hid_dim": 32, "order": 2, "cpu": False},
    {"label": "gpu, lower order",      "hid_dim": 32, "order": 1, "cpu": False},
    {"label": "cpu, published config", "hid_dim": 64, "order": 2, "cpu": True},
    {"label": "cpu, half width",       "hid_dim": 32, "order": 2, "cpu": True},
]


def build_cmd(model, rung, seeds, epochs, trials, tune_epochs, results_root,
              probe=False):
    cmd = [sys.executable, "scripts/run_benchmark.py",
           "--models", model, "--datasets", "tsocial",
           "--ratios", "TR40", "--seeds", *[str(s) for s in seeds],
           "--trials", str(trials),
           "--tune-epochs", str(tune_epochs), "--tune-patience", "4",
           "--epochs", str(epochs), "--patience", str(FINAL_PATIENCE),
           "--max-minutes", "20" if probe else "240",
           "--data-root", "data", "--results-root", results_root,
           "--no-archive", "--no-report",
           # Forced on tuning trials and final runs alike, and written into
           # every run_config.yml, so the reported number always carries the
           # configuration that produced it.
           "--override", f"hid_dim={rung['hid_dim']}", f"order={rung['order']}"]
    if model == "CARE-GNN":
        cmd.append("--allow-large")
    if rung["cpu"]:
        cmd.append("--no-cuda")
    return cmd


def run_cmd(cmd, show=("trial", "COMPLETE", "FAILED", "seed", "out of memory",
                       "OutOfMemory", "isolation")):
    proc = subprocess.Popen(cmd, cwd=str(WORK), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        tail.append(line)
        if any(k in line for k in show):
            print("   ", line.rstrip())
    proc.wait()
    return proc.returncode, "".join(tail[-40:])


def find_rung(model):
    """Probe each rung for two epochs; return the first that does not OOM."""
    for index, rung in enumerate(LADDER, 1):
        where = "CPU" if rung["cpu"] else "GPU"
        print(f"\n   -- rung {index}/{len(LADDER)}: {rung['label']} "
              f"(hid_dim={rung['hid_dim']}, order={rung['order']}, {where})")
        probe_root = f"probe_{model.lower().replace('-','')}_{index}"
        code_, tail = run_cmd(build_cmd(model, rung, [2], 2, 1, 2, probe_root,
                                        probe=True))
        oom = "OutOfMemory" in tail or "out of memory" in tail
        got = list((WORK / probe_root).rglob("test_metrics.json"))

        # The tuning log prints the grid's candidate, not the forced values, so
        # read back what the run actually used. Without this it is impossible to
        # tell a working override from a silently ignored one.
        applied = None
        for path in (WORK / probe_root).rglob("summary.json"):
            try:
                cfg = json.loads(path.read_text(encoding="utf-8"))["configuration"]
                applied = (cfg.get("hid_dim"), cfg.get("order"),
                           cfg.get("resolved_device"))
                break
            except Exception as error:
                print(f"      (could not read {path.name}: {error})")
        if applied:
            print(f"      applied: hid_dim={applied[0]}, order={applied[1]}, "
                  f"device={applied[2]}")
            if applied[0] != rung["hid_dim"] or applied[1] != rung["order"]:
                print("      WARNING: the override did not take effect; this "
                      "rung tested a different configuration than requested")
        else:
            # No record at all means the command never ran - an argument error,
            # a missing flag - rather than a run that failed on its merits.
            # Those look identical in the log otherwise, and one wasted two
            # rungs of this ladder already.
            print("      no run record written: the command did not start")
            for line in tail.strip().splitlines()[-4:]:
                print(f"        {line.strip()[:110]}")

        shutil.rmtree(WORK / probe_root, ignore_errors=True)
        if code_ == 0 and got and not oom:
            print(f"   -> rung {index} fits")
            return rung
        print(f"   -> rung {index} {'ran out of memory' if oom else 'did not complete'}")
    return None

## 3. Run

The chosen rung is used for the full protocol budget: 100 epochs, patience 20,
three seeds, one test evaluation each.

In [5]:
def full_run(model):
    print(f"\n{'='*70}\n{model} x T-Social\n{'='*70}")
    rung = find_rung(model)
    if rung is None:
        print(f"\n   No configuration fits on this hardware for {model}.")
        print("   This is a real, reportable outcome: T-Social needs a larger")
        print("   accelerator. The probe failures are recorded with tracebacks.")
        return False
    print(f"\n   running the full protocol at: {rung['label']} "
          f"(hid_dim={rung['hid_dim']}, order={rung['order']}, "
          f"{'CPU' if rung['cpu'] else 'GPU'})")
    code_, _ = run_cmd(build_cmd(model, rung, SEEDS, FINAL_EPOCHS, TRIALS,
                                 TUNE_EPOCHS, "results"))
    print(f"   exit {code_}")
    return code_ == 0

if DO_GHRN:
    step("GHRN x T-Social")(lambda: full_run("GHRN"))
if DO_CARE_GNN:
    step("CARE-GNN x T-Social")(lambda: full_run("CARE-GNN"))


GHRN x T-Social

GHRN x T-Social

   -- rung 1/5: gpu, published config (hid_dim=64, order=2, GPU)
    torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 1.38 GiB. GPU 0 has a total capacity of 14.56 GiB of which 638.81 MiB is free. Including non-PyTorch memory, this process has 13.94 GiB memory in use. Of the allocated memory 13.73 GiB is allocated by PyTorch, and 86.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
    GHRN | tsocial | unified | TR40 | seed 2
    Run ID: ghrn_tsocial_unified_tr40_seed2_t4_20260917T100407Z
    RUN FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 1.38 GiB. GPU 0 has a total capacity of 14.56 GiB of which 638.81 MiB is free. Including non-PyTorch memory, this process has 13.94 GiB memory in 

## 4. Package

In [7]:
@step("package")
def _():
    if not (WORK / "results").exists():
        print("   Nothing trained. Every rung failed, which is the result:")
        print("   T-Social does not fit this hardware at any legal configuration.")
        print("   Report it as FAILED_TECHNICAL with the recorded tracebacks.")
        return False
    subprocess.run([sys.executable, "scripts/build_deliverable.py",
                    "--results", "results",
                    "--out", str(OUTDIR / "DELIVERABLE_TSOCIAL"),
                    "--platform", "Kaggle (Tesla T4)"],
                   capture_output=True, text=True, cwd=str(WORK))
    shutil.make_archive(str(OUTDIR / "tsocial_results"), "zip", str(WORK / "results"))
    return True

print(f"\n{'='*70}\nSUMMARY\n{'='*70}")
for name, info in STATUS.items():
    mark = "OK  " if info["ok"] else "FAIL"
    print(f"  {mark}  {name:<28} {info['minutes']:6.1f} min"
          + ("" if info["ok"] else f"  {info['error'][:55]}"))

done = []
if (WORK / "results").exists():
    for p in (WORK / "results").rglob("summary.json"):
        try:
            j = json.loads(p.read_text())
        except Exception:
            continue
        if j.get("test_metrics") and not str(
                j.get("configuration", {}).get("run_mode", "")).startswith("tuning"):
            done.append((j["model"], j["train_seed"],
                         j["test_metrics"].get("auroc"),
                         j["configuration"].get("hid_dim"),
                         j["configuration"].get("order")))

print(f"\ncompleted T-Social runs: {len(done)}")
for model, seed, auroc, hid, order in sorted(done):
    print(f"  {model:<9} seed {seed:<3} AUROC {auroc:.4f}  "
          f"(hid_dim={hid}, order={order})")

print("\nfiles to download:")
for f in sorted(OUTDIR.glob("*.zip")):
    print(f"  {f.name:<34} {f.stat().st_size/1024**2:8.1f} MB")


package
   Nothing trained. Every rung failed, which is the result:
   T-Social does not fit this hardware at any legal configuration.
   Report it as FAILED_TECHNICAL with the recorded tracebacks.

SUMMARY
  OK    1. unpack pipeline              0.0 min
  OK    2. torch + DGL                  3.2 min
  OK    3. download and decode T-Social    4.9 min
  OK    2a. verify int32 narrowing      0.5 min
  OK    GHRN x T-Social                53.0 min
  OK    package                         0.0 min

completed T-Social runs: 0

files to download:


In [8]:
import shutil

shutil.make_archive(
    "/kaggle/working/comp8851",
    "zip",
    "/kaggle/working/comp8851"
)

print("Created: /kaggle/working/comp8851.zip")


Created: /kaggle/working/comp8851.zip


In [10]:
from IPython.display import FileLink

FileLink("/kaggle/working/comp8851.zip")


/kaggle/working/comp8851.zip